# PMAPS Workshop: IDAES-GTEP, Session 2

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/agmoore4/idaes-gtep.git/pmaps-123?urlpath=%2Fdoc%2Ftree%2Fdocs%2Fsource%2Ftutorials%2F123bus%2Ftutorial_123bus.ipynb)

Welcome! In this tutorial, we will demonstrate using IDAES Generation and Transmission Expansion Planning (GTEP) with a 123-bus system in Texas.

As we step through the notebook, you should notice that all the steps to set up and solve a model are the same as in the simpler 5-bus case. The only major difference is the scale of the system reflected in the data files. However, we will demonstrate some helpful (optional) functionality for larger cases that take longer to solve.

In [1]:
# We'll again need to install a solver. Run this cell, then restart your notebook and continue.
%pip install highspy

Looking in indexes: https://nexus.web.sandia.gov/repository/pypi-group/simple/
Note: you may need to restart the kernel to use updated packages.


In [2]:
# suppressing some logs/warnings
import logging
import warnings
logging.getLogger().setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

## Setting up and solving the model

First, we create our `ExpansionPlanningData` instance and use `load_prescient` to load the relevant data files.

In [3]:
from pathlib import Path
from gtep.gtep_data import ExpansionPlanningData

data_path = Path("../../../../gtep/data/123_Bus_Resil_Week")

rep_days=[
    "2019-01-28 00:00",
    # "2019-03-07 00:00",
    "2019-03-16 00:00",
    # "2019-05-10 00:00",
    "2019-06-10 00:00",
    # "2019-06-30 00:00",
    "2019-07-15 00:00",
    # "2019-10-29 00:00",
    "2019-11-03 00:00",
    # "2019-12-10 00:00",
]
rep_weights={
    "2019-01-28 00:00": 80,
    # "2019-03-07 00:00": 35,
    "2019-03-16 00:00": 60,
    # "2019-05-10 00:00": 35,
    "2019-06-10 00:00": 65,
    # "2019-06-30 00:00": 35,
    "2019-07-15 00:00": 80,
    # "2019-10-29 00:00": 35,
    "2019-11-03 00:00": 80,
    # "2019-12-10 00:00": 40,
} 

data_object = ExpansionPlanningData(
    stages=1,
    num_reps=5,
    num_commit=1,
    num_dispatch=1,
    duration_representative_period=1,
)
data_object.load_prescient(
    data_path,
    representative_dates=rep_days,
    representative_weights=rep_weights,
)

Interactive Python mode detected; using default matplotlib backend for plotting.
Setting default t0 state in RTS-GMLC parser


Next we read in cost data using `DataProcessing`:

In [4]:
from gtep.gtep_data_processing import DataProcessing

bus_data_path = Path(
    "../../../../gtep/data/costs/Bus_data_gen_weights_mappings.csv"
)
cost_data_path = Path(
    "../../../../gtep/data/costs/2022_v3_Annual_Technology_Baseline_Workbook_Mid-year_update_2-15-2023_Clean.xlsx"
)
ng_cost_path = Path(
    "../../../../gtep/data/costs/Total_Energy_Supply_Disposition_and_Price_Summary.csv"
)

candidate_gens = [
    "Natural Gas_FE",
    "Solar - Utility PV",
    "Land-Based Wind",
]

cost_data = DataProcessing()
cost_data.load_gen_data(
    bus_data_path=bus_data_path,
    cost_data_path=cost_data_path,
    ng_cost_path=ng_cost_path,
    candidate_gens=candidate_gens,
)

Finally, we create and solve the model. However, we'll do two small (optional) things differently since the 123-bus case takes longer to solve:

1. Since the 123-bus case is much larger than the 5-bus case, it can be useful to track how much time elapses between each step (build, transform, and solve). To do so, we can make use of the `ExpansionPlanningModel`'s `.timer` attribute, which is a `TicTocTimer` instance (see the Pyomo docs for it here: https://pyomo.readthedocs.io/en/stable/api/pyomo.common.timing.TicTocTimer.html). In short, calling `mod_object.timer.toc()` will report the elapsed time since the last `.tic()`/`.toc()`. Note that an `ExpansionPlanningModel` automatically calls `self.timer.tic()` when `.create_model()` is called on it (with the message `"Creating GTEP Model"`).
2. Passing configuration options to the solver is a good way to help the model solve faster. For the sake of the tutorial, we pass `opt.options["mip_rel_gap"] = 0.01` to HiGHS below, which allows the solver to terminate once the gap between the current best solution and lower bound is under 1%. Other useful options include scaling (such as `user_objective_scale` and `user_bound_scale`).

In total, the cell below should take ~3 minutes to run.

In [ ]:
from gtep.gtep_model import ExpansionPlanningModel
from pyomo.environ import SolverFactory, TransformationFactory
from contextlib import redirect_stdout, redirect_stderr

mod_object = ExpansionPlanningModel(
    data=data_object,
    cost_data=cost_data,
    config={"scale_loads": False},
)
mod_object.create_model()
mod_object.timer.toc("Finished model build")

TransformationFactory("gdp.bigm").apply_to(mod_object.model)
mod_object.timer.toc("Finished model transformation")

opt = SolverFactory("highs")
opt.options["mip_rel_gap"] = 0.01

with open("output.log", "w") as f, redirect_stdout(f), redirect_stderr(f):
    result = opt.solve(mod_object.model, tee=True)

mod_object.timer.toc("Finished solving");

[    0.00] Creating GTEP Model
[+   1.53] Finished model build
[+   1.87] Finished model transformation
Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 37884 rows; 25172 cols; 101635 nonzeros; 12350 integer variables (11871 binary)
Coefficient ranges:
  Matrix  [1e+00, 3e+06]
  Cost    [1e+00, 4e+09]
  Bound   [1e+00, 4e+03]
  RHS     [3e-01, 5e+07]
Presolving model
20186 rows, 14698 cols, 55507 nonzeros 0s
17158 rows, 12173 cols, 46407 nonzeros 0s
17148 rows, 12163 cols, 46507 nonzeros 0s
Presolve reductions: rows 17148(-20736); columns 12163(-13009); nonzeros 46507(-55128) 

Solving MIP model with:
   17148 rows
   12163 cols (7517 binary, 0 integer, 0 implied int., 4646 continuous, 0 domain fixed)
   46507 nonzeros
   Thread count 4 (of 8 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump

Now we use the `ExpansionPlanningSolution` class to write out the results and generate plots:

In [10]:
from copy import copy
from IPython.display import display, HTML
from gtep.gtep_solution import ExpansionPlanningSolution

def display_plotly_as_html(fig, title=None):
    if title is not None:
        fig = copy(fig)
        fig.update_layout(title=title)
    fig_html = fig.to_html(full_html=False, include_plotlyjs="cdn")
    return display(HTML(fig_html))

# create solution object and write out to json
soln = ExpansionPlanningSolution(data_path)
soln_path = Path("soln")
soln.save_results_in_json_files(mod_object, soln_path)

# perform plotting
pie = soln.create_plots("combined", soln_path, data_path, "piechart", savefig=False)
stackgraph = soln.create_stackgraph(soln_path, rep_days, savefig=False)

display_plotly_as_html(pie, title="Baseline 123-bus case")
display_plotly_as_html(stackgraph, title="Baseline 123-bus case")

[Markdown discussing outputs here]

## Experiments

Here we can again define a helper function to solve the model and return plots:

In [12]:
from pyomo.environ import value

def solve_model_and_make_plots(
    model_object: ExpansionPlanningModel,
    data_path: Path|str,
    write_dir: Path|str,
    tee: bool=True,
    solver_options: None|dict=None,
):
    
    # transform and solve
    TransformationFactory("gdp.bigm").apply_to(model_object.model)
    model_object.timer.toc("Finished model transformation")

    opt = SolverFactory("highs")
    if solver_options is not None:
        for option, val in solver_options.items():
            opt.options[option] = val

    result_object = opt.solve(model_object.model, tee=tee)
    model_object.timer.toc("Finished solving")

    # check termination condition
    term_cond = result_object["Solver"][0]["Termination condition"]
    print("Termination condition:", term_cond)
    if term_cond != "optimal":
        return

    # make solution
    soln_object = ExpansionPlanningSolution(data_path)
    soln_object_path = (Path() / write_dir).resolve()
    soln_object.save_results_in_json_files(model_object, soln_object_path)
    model_object.timer.toc("Finished saving results to json")
    pie = soln_object.create_plots(
        "combined", soln_object_path, data_path, "piechart", savefig=False
    )
    rep_days = [
        value(model_object.model.representativeDate[idx])
        for idx in model_object.model.representativeDate
    ]
    stackgraph = soln_object.create_stackgraph(
        soln_object_path, rep_days, savefig=False
    )
    model_object.timer.toc("Finished plotting")
    return pie, stackgraph

### Experiment 1

In this experiment, we keep the number of time periods the same, but we set `include_investment=False`. This means that candidate assets cannot be invested in, and the model must do what it can with what is already installed.

This cell will take ~2 minutes to run.

In [15]:
mod_object_expr1 = ExpansionPlanningModel(
    data=data_object,  # same data object
    cost_data=cost_data,  # same cost data object
    config={"scale_loads": False, "include_investment": False},
)
mod_object_expr1.create_model()
mod_object_expr1.timer.toc("Finished model build")

# for this experiment, we'll just look at the pie chart
pie_exper1, _ = solve_model_and_make_plots(
    mod_object_expr1,
    data_path,
    "soln_expr1",
    solver_options={"mip_rel_gap": 0.01}
)

print(
    "Objective for baseline:",
    f"{value(mod_object.model.total_cost_objective) :,.0f}"
)
print(
    "Objective for experiment 1:",
    f"{value(mod_object_expr1.model.total_cost_objective) :,.0f}"
)

display_plotly_as_html(pie, title="Baseline")
display_plotly_as_html(
    pie_exper1,
    title="Experiment 1 (no investment)",
)

[    0.00] Creating GTEP Model
[+   1.38] Finished model build
[+   2.30] Finished model transformation
Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 37883 rows; 25172 cols; 96093 nonzeros; 12350 integer variables (11754 binary)
Coefficient ranges:
  Matrix  [1e+00, 3e+06]
  Cost    [1e+00, 4e+09]
  Bound   [1e+00, 4e+03]
  RHS     [3e-01, 5e+07]
Presolving model
12095 rows, 9919 cols, 34809 nonzeros 0s
11162 rows, 8136 cols, 32380 nonzeros 0s
Presolve reductions: rows 11162(-26721); columns 8136(-17036); nonzeros 32380(-63713) 

Solving MIP model with:
   11162 rows
   8136 cols (4475 binary, 0 integer, 0 implied int., 3661 continuous, 0 domain fixed)
   32380 nonzeros
   Thread count 4 (of 8 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feas

[Discuss stuff]

### Experiment 2

transmission=True

In [20]:
mod_object_expr2 = ExpansionPlanningModel(
    data=data_object,  # same data object
    cost_data=cost_data,  # same cost data object
    config={"scale_loads": False, "transmission": True},
)
mod_object_expr2.create_model()
mod_object_expr2.timer.toc("Finished model build")

# for this experiment, we'll just look at the pie chart
pie_exper2, stackgraph_exper2 = solve_model_and_make_plots(
    mod_object_expr2,
    data_path,
    "soln_expr2",
    solver_options={"mip_rel_gap": 0.01}
)

print(
    "Objective for baseline:",
    f"{value(mod_object.model.total_cost_objective) :,.0f}"
)
print(
    "Objective for experiment 2:",
    f"{value(mod_object_expr1.model.total_cost_objective) :,.0f}"
)

display_plotly_as_html(pie, title="Baseline")
display_plotly_as_html(
    pie_exper2,
    title="Experiment 2 (with transmission investment)",
)
display_plotly_as_html(stackgraph, title="Baseline")
display_plotly_as_html(
    stackgraph_exper2,
    title="Experiment 2 (with transmission investment)",
)

[    0.00] Creating GTEP Model
[+   1.35] Finished model build
[+   1.89] Finished model transformation
Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 57264 rows; 31547 cols; 142180 nonzeros; 18725 integer variables (17991 binary)
Coefficient ranges:
  Matrix  [1e+00, 3e+06]
  Cost    [1e+00, 1e+10]
  Bound   [1e+00, 4e+03]
  RHS     [3e-01, 5e+07]
Presolving model
19370 rows, 14069 cols, 52073 nonzeros 0s
15725 rows, 11850 cols, 41362 nonzeros 0s
14166 rows, 10107 cols, 37656 nonzeros 0s
Presolve reductions: rows 14166(-43098); columns 10107(-21440); nonzeros 37656(-104524) 

Solving MIP model with:
   14166 rows
   10107 cols (6888 binary, 0 integer, 0 implied int., 3219 continuous, 0 domain fixed)
   37656 nonzeros
   Thread count 4 (of 8 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pum

[Discuss results]

### Experiment 3

In this experiment, we showcase solving the model for many more time periods [describe specifics here].

The model takes several hours to run, so we will show how to set it up but won't solve it here. Instead, we will load solution jsons from a previous run to produce figures.

In [23]:
data_object_exper3 = ExpansionPlanningData(
    stages=1,
    num_reps=10,
    num_commit=5,
    num_dispatch=1,
    duration_representative_period=10,
)

data_object_exper3.load_prescient(
    data_path,
    representative_dates=[
        "2019-01-28 00:00",
        "2019-03-07 00:00",
        "2019-03-16 00:00",
        "2019-05-10 00:00",
        "2019-06-10 00:00",
        "2019-06-30 00:00",
        "2019-07-15 00:00",
        "2019-10-29 00:00",
        "2019-11-03 00:00",
        "2019-12-10 00:00",
    ],
    representative_weights={
        "2019-01-28 00:00": 45,
        "2019-03-07 00:00": 35,
        "2019-03-16 00:00": 25,
        "2019-05-10 00:00": 35,
        "2019-06-10 00:00": 30,
        "2019-06-30 00:00": 35,
        "2019-07-15 00:00": 45,
        "2019-10-29 00:00": 35,
        "2019-11-03 00:00": 40,
        "2019-12-10 00:00": 40,
    },
)

mod_object_expr3 = ExpansionPlanningModel(
    data=data_object_exper3,
    cost_data=cost_data,  # same cost data
    config={"scale_loads": False, "transmission": True},
)
mod_object_expr3.create_model()
mod_object_expr3.timer.toc("Finished model build")

print("-" * 50)
print("We then could continue on to solve...")

Setting default t0 state in RTS-GMLC parser
[    0.00] Creating GTEP Model
[+  19.03] Finished model build
--------------------------------------------------
We then could continue on to solve...


In [24]:
saved_result_dir = "soln_expr3"

soln_object_exper3 = ExpansionPlanningSolution(data_path)
soln_object_path_exper3 = (Path() / saved_result_dir).resolve()

pie_exper3 = soln_object_exper3.create_plots(
    "combined", soln_object_path_exper3, data_path, "piechart", savefig=False
)
rep_days_expr3 = [
    value(mod_object_expr3.model.representativeDate[idx])
    for idx in mod_object_expr3.model.representativeDate
]
stackgraph_exper3 = soln_object_exper3.create_stackgraph(
    soln_object_path_exper3, rep_days_expr3, savefig=False
)

display_plotly_as_html(pie, title="Baseline")
display_plotly_as_html(pie_exper3, title="Experiment 3 (bigger model)")

display_plotly_as_html(stackgraph, title="Baseline")
display_plotly_as_html(stackgraph_exper3, title="Experiment 3 (bigger model)")

[Discuss results here]